In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

#Smart file locator
if os.path.exists("data/data/SPY_15m_data.csv"):
    target_file = "data/data/SPY_15m_data.csv"
else:
    target_file = "data/SPY_15m_data.csv"

#Load the data 
data = pd.read_csv(target_file, index_col=0, parse_dates=True)

#Rebuild the core baselines for comparison
data['market_return'] = data['close'].pct_change()
data['buy_and_hold'] = (1 + data['market_return']).cumprod()

#Clean up the first empty row
data.dropna(inplace=True)

print(f"Environment ready! Data loaded and baselines calculated. Total rows: {len(data)}")

Environment ready! Data loaded and baselines calculated. Total rows: 1559


In [11]:
#Parameters
ma_length = 20
stop_loss_pct = 0.005

#Indicator
data['SMA_20'] = data['close'].rolling(window=ma_length).mean()

#Logic and Return
data['raw_signal'] = np.where(data['close'] > data['SMA_20'], 1, -1)
data['raw_strat_return'] = data['raw_signal'].shift(1) * data['market_return']
data['strat_return_with_sl'] = np.clip(data['raw_strat_return'], -stop_loss_pct, np.inf)
data['single_ma_cumulative'] = (1 + data['strat_return_with_sl']).cumprod()

#Results
final_return = (data['single_ma_cumulative'].iloc[-1] - 1) * 100
buy_hold_return = (data['buy_and_hold'].iloc[-1] - 1) * 100

print("--- SINGLE MA (20-Period) RESULTS ---")
print(f"Buy & Hold Baseline: {buy_hold_return:.2f}%")
print(f"Strategy Return (with {stop_loss_pct*100}% Stop-Loss): {final_return:.2f}%")


--- SINGLE MA (20-Period) RESULTS ---
Buy & Hold Baseline: 5.40%
Strategy Return (with 0.5% Stop-Loss): -0.85%
